<style>
  .nb-section {
    direction: rtl;
    text-align: right;
    font-family: "Arial", "Tahoma", sans-serif;
    line-height: 1.9;
    color: #e8eaf0;
    background: linear-gradient(135deg, #0b1020 0%, #111827 100%);
    border: 1px solid #26324a;
    border-radius: 14px;
    padding: 18px 22px;
    margin: 12px 0;
  }

  .nb-section h1,
  .nb-section h2,
  .nb-section h3 {
    margin-top: 0;
    color: #ffffff;
    font-weight: 800;
  }

  .nb-section h1 {
    font-size: 30px;
    border-bottom: 1px solid #334155;
    padding-bottom: 10px;
  }

  .nb-section h2 {
    font-size: 24px;
  }

  .nb-section h3 {
    font-size: 19px;
    color: #93c5fd;
  }

  .nb-section p {
    margin: 8px 0;
    font-size: 15.5px;
  }

  .nb-section ul {
    margin-top: 8px;
    padding-right: 24px;
  }

  .nb-section li {
    margin: 6px 0;
  }

  .ltr {
    direction: ltr;
    unicode-bidi: isolate;
    display: inline-block;
    font-family: "Consolas", "Courier New", monospace;
    color: #7dd3fc;
    background: rgba(14, 165, 233, 0.10);
    padding: 1px 6px;
    border-radius: 6px;
  }

  .metric {
    direction: ltr;
    unicode-bidi: isolate;
    display: inline-block;
    color: #c4b5fd;
    font-family: "Consolas", "Courier New", monospace;
    background: rgba(168, 85, 247, 0.12);
    padding: 1px 6px;
    border-radius: 6px;
  }

  .note {
    background: rgba(59, 130, 246, 0.10);
    border-right: 4px solid #38bdf8;
    padding: 10px 14px;
    border-radius: 10px;
    margin-top: 12px;
  }

  .warning {
    background: rgba(245, 158, 11, 0.12);
    border-right: 4px solid #f59e0b;
    padding: 10px 14px;
    border-radius: 10px;
    margin-top: 12px;
  }

  .success {
    background: rgba(34, 197, 94, 0.12);
    border-right: 4px solid #22c55e;
    padding: 10px 14px;
    border-radius: 10px;
    margin-top: 12px;
  }

  .nb-table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 12px;
    direction: rtl;
  }

  .nb-table th,
  .nb-table td {
    border: 1px solid #334155;
    padding: 9px 10px;
    text-align: right;
  }

  .nb-table th {
    background: rgba(59, 130, 246, 0.18);
    color: #ffffff;
  }

  .nb-table td {
    background: rgba(15, 23, 42, 0.65);
  }
</style>

<div class="nb-section">
  <h1>News Comment Topic Analysis System</h1>
  <h2>Final BERTopic Evaluation Notebook</h2>

  <p>
    يقدّم هذا الدفتر التقييم النهائي لنموذج
    <span class="ltr">Final Fine-Tuned MiniLM Comment Encoder</span>
    داخل خط معالجة
    <span class="ltr">BERTopic</span>
    لاستخراج المواضيع من تعليقات الأخبار.
  </p>

  <p>
    يركّز الدفتر على تشغيل خط التقييم النهائي فقط، بدءًا من بيانات التعليقات النظيفة،
    ثم توليد
    <span class="ltr">embeddings</span>،
    ثم اختزال الأبعاد باستخدام
    <span class="ltr">UMAP</span>،
    ثم التجميع باستخدام
    <span class="ltr">HDBSCAN</span>،
    وصولًا إلى استخراج المواضيع وقياس جودة النتائج.
  </p>

  <div class="note">
    لا يعرض هذا الدفتر مراحل التجارب الداخلية السابقة، بل يعرض النموذج النهائي المعتمد
    وخط التقييم القابل لإعادة التشغيل.
  </div>
</div>

<div dir="rtl" style="text-align:right; line-height:1.8; font-size:16px;">

<h2>01 - Environment Setup</h2>

<p>
في هذه الخلية نقوم بتثبيت المكتبات المطلوبة لتشغيل نظام
<span dir="ltr">Topic Modeling</span>
باستخدام
<span dir="ltr">BERTopic</span>
ونموذج
<span dir="ltr">SentenceTransformer</span>
المدرّب مسبقًا في المشروع.
</p>

<p>
تم تضمين مكتبة
<span dir="ltr">NLTK</span>
لاستخدام قائمة
<span dir="ltr">English Stopwords</span>
لأن نصوص المشروع باللغة الإنجليزية وليست باللغة العربية.
</p>

</div>

In [ ]:
!pip install -q bertopic==0.16.0 sentence-transformers umap-learn hdbscan scikit-learn pandas numpy matplotlib nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.1/154.1 kB 6.8 MB/s eta 0:00:00


<div class="nb-section">
  <h2>02 — Import Libraries</h2>

  <p>
    تحتوي هذه الخلية على الاستيرادات اللازمة للتعامل مع البيانات، الملفات،
    النماذج، توليد
    <span class="ltr">embeddings</span>،
    بناء
    <span class="ltr">BERTopic</span>،
    وحساب المقاييس النهائية.
  </p>

  <p>
    تُستخدم مكتبات
    <span class="ltr">NumPy</span>
    و
    <span class="ltr">Pandas</span>
    لمعالجة البيانات، بينما تُستخدم
    <span class="ltr">SentenceTransformer</span>
    لتوليد تمثيلات التعليقات.
  </p>
</div>

In [ ]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

import nltk
from nltk.corpus import stopwords

<div class="nb-section">
  <h2>03 — Reproducibility Settings</h2>

  <p>
    يتم تثبيت قيمة
    <span class="ltr">random_state = 42</span>
    لضمان تقليل التغيّر بين مرات التشغيل، خصوصًا في خطوات مثل
    <span class="ltr">UMAP</span>
    وبعض العمليات العشوائية داخل خط المعالجة.
  </p>

  <div class="warning">
    بعض مراحل
    <span class="ltr">GPU</span>
    أو الخوارزميات التقريبية قد تنتج اختلافات طفيفة بين تشغيل وآخر،
    لكن تثبيت البذرة العشوائية يجعل النتائج أكثر قابلية لإعادة الإنتاج.
  </div>
</div>

In [ ]:
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

<div class="nb-section">
  <h2>04 — Mount Google Drive</h2>

  <p>
    تربط هذه الخلية بيئة
    <span class="ltr">Google Colab</span>
    مع
    <span class="ltr">Google Drive</span>
    للوصول إلى ملفات المشروع، نموذج
    <span class="ltr">embedding</span>
    النهائي، وملفات بيانات التقييم.
  </p>

  <p>
    جميع المسارات المستخدمة في الدفتر مبنية على مجلد المشروع داخل
    <span class="ltr">Google Drive</span>.
  </p>
</div>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

# Original final model location
SOURCE_MODEL_DIR = PROJECT_ROOT / "models/v7_fullft_minilm_raw_largepairs_1p5M_seq256/final_model"

# Clean academic model alias
CLEAN_MODEL_DIR = PROJECT_ROOT / "models/final_finetuned_minilm_comment_encoder"

if not SOURCE_MODEL_DIR.exists():
    raise FileNotFoundError(f"Source model not found: {SOURCE_MODEL_DIR}")

if CLEAN_MODEL_DIR.exists():
    print("Clean model directory already exists:", CLEAN_MODEL_DIR)
else:
    shutil.copytree(SOURCE_MODEL_DIR, CLEAN_MODEL_DIR)
    print("Copied final model to:", CLEAN_MODEL_DIR)


# Original evaluation data location
SOURCE_DATA_PATH = PROJECT_ROOT / "training_data/v5_clean_pools_20260420_194922/test_large_clean_pool.csv"

# Clean academic evaluation data alias
CLEAN_DATA_DIR = PROJECT_ROOT / "evaluation_data/final_test_pool"
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_DATA_PATH = CLEAN_DATA_DIR / "test_large_clean_pool.csv"

if not SOURCE_DATA_PATH.exists():
    raise FileNotFoundError(f"Source data not found: {SOURCE_DATA_PATH}")

if CLEAN_DATA_PATH.exists():
    print("Clean data file already exists:", CLEAN_DATA_PATH)
else:
    shutil.copy2(SOURCE_DATA_PATH, CLEAN_DATA_PATH)
    print("Copied evaluation data to:", CLEAN_DATA_PATH)

Copied final model to: /content/drive/MyDrive/semester project/news_comment_topic_system/models/final_finetuned_minilm_comment_encoder
Copied evaluation data to: /content/drive/MyDrive/semester project/news_comment_topic_system/evaluation_data/final_test_pool/test_large_clean_pool.csv


<div class="nb-section">
  <h2>05 — Project Paths</h2>

  <p>
    تحدد هذه الخلية المسارات الأساسية المستخدمة في التقييم النهائي:
  </p>

  <ul>
    <li>مسار ملف بيانات التقييم النظيفة.</li>
    <li>مسار النموذج النهائي المعتمد.</li>
    <li>مسار حفظ مخرجات <span class="ltr">BERTopic</span>.</li>
  </ul>

  <p>
    الاسم المعتمد للنموذج في هذا الدفتر هو:
    <span class="ltr">Final Fine-Tuned MiniLM Comment Encoder</span>.
  </p>

  <div class="note">
    تم استخدام اسم نهائي واضح بدل أسماء التجارب الداخلية، حتى يكون الدفتر مباشرًا
    ومناسبًا للمراجعة الأكاديمية.
  </div>
</div>

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

FINAL_MODEL_DISPLAY_NAME = "Final Fine-Tuned MiniLM Comment Encoder"
FINAL_EXPERIMENT_NAME = "final_minilm_comment_encoder_umap15_mcs100_ms10"

DATA_PATH = PROJECT_ROOT / "evaluation_data/final_test_pool/test_large_clean_pool.csv"

MODEL_PATH = PROJECT_ROOT / "models/final_finetuned_minilm_comment_encoder"

OUTPUT_DIR = PROJECT_ROOT / "final_bertopic_evaluation_outputs" / FINAL_EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Final model:", FINAL_MODEL_DISPLAY_NAME)
print("DATA_PATH exists:", DATA_PATH.exists())
print("MODEL_PATH exists:", MODEL_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

Final model: Final Fine-Tuned MiniLM Comment Encoder
DATA_PATH exists: True
MODEL_PATH exists: True
OUTPUT_DIR: /content/drive/MyDrive/semester project/news_comment_topic_system/final_bertopic_evaluation_outputs/final_minilm_comment_encoder_umap15_mcs100_ms10


<div class="nb-section">
  <h2>06 — Load Evaluation Dataset</h2>

  <p>
    تقوم هذه الخلية بتحميل مجموعة بيانات التقييم النهائية إلى
    <span class="ltr">DataFrame</span>
    باستخدام مكتبة
    <span class="ltr">Pandas</span>.
  </p>

  <p>
    يتم عرض شكل البيانات وأسماء الأعمدة للتأكد من أن ملف التقييم تم تحميله بشكل صحيح
    قبل بدء خطوات تنظيف النصوص وتوليد
    <span class="ltr">embeddings</span>.
  </p>
</div>

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
print(df.columns.tolist())

Shape: (58014, 27)


,date,user_id,user_name,text,raw_post_name,post_id_true,post_text,post_description,post_link,post_page_id,...,post_react_sad,post_react_wow,raw_post_id,has_post_context,split,normalized_text,clean_tokens,clean_token_count,informative_token_count,clean_text
0,2017-07-10 19:33:45+00:00,d6b2adafdf22ca873972,1648dec1dc56d2a49b6e,NO NO THEY \r\nRANSACKED THE PRISONS AND FREED...,d6b2adafdf22ca873972_10155767377238690,10155767377238690,This 12-year-old boy was recruited by ISIL in ...,NaN,https://www.facebook.com/aljazeera/videos/1015...,7.382474e+09,...,268.0,20.0,7382473689_10155767377238690,True,test,no no they ransacked the prisons and freed the...,"['they', 'ransacked', 'the', 'prisons', 'and',...",12,11,they ransacked the prisons and freed them that...
1,2017-07-12 20:12:14+00:00,94c0d5555aec967f5f1a,34df1f2ff9ba32ce4236,stupidity is what Alex relies on to sell his p...,94c0d5555aec967f5f1a_10155578714737577,10155578714737577,Google’s scheme of paying professors to influe...,"Professors get funding to push ""global warming...",https://www.infowars.com/google-expose-reveals...,8.025673e+10,...,4.0,30.0,80256732576_10155578714737577,True,test,stupidity is what alex relies on to sell his p...,"['stupidity', 'what', 'alex', 'relies', 'sell'...",9,9,stupidity what alex relies sell his products f...
2,2017-07-13 16:05:17+00:00,9088c14541bd47a39d06,d6ccfe1f43ab991eb634,"""Silver, once one of the most powerful people ...",9088c14541bd47a39d06_10154838273985950,10154838273985950,"Silver, once one of the most powerful people i...",,http://cbsn.ws/2tQkH5Z,1.314593e+11,...,4.0,21.0,131459315949_10154838273985950,True,test,silver once one of the most powerful people in...,"['silver', 'once', 'one', 'the', 'most', 'powe...",35,32,silver once one the most powerful people new y...
3,2017-07-06 20:08:04+00:00,1127ab6051b5310e9c06,b40d7812d2fe4c71032e,Democrats have placed such emphasis and relied...,1127ab6051b5310e9c06_10156565672364657,10156565672364657,"Here we go, again.",,http://trib.al/oPv8jiy,4.163279e+10,...,1.0,5.0,41632789656_10156565672364657,True,test,democrats have placed such emphasis and relied...,"['democrats', 'have', 'placed', 'such', 'empha...",96,96,democrats have placed such emphasis and relied...
4,2017-07-05 19:59:04+00:00,fbbfc9141b8520e4447c,1f5ad6bd4b8a29683c5a,This is why we can't have nice things.\r\nhttp...,fbbfc9141b8520e4447c_10155053452139160,10155053452139160,"""CNN has screwed up royally this time.""\n\nThe...","The news group published a report Tuesday, tit...",http://washex.am/2tR95lY,4.065670e+10,...,3.0,17.0,40656699159_10155053452139160,True,test,this is why we can not have nice things,"['this', 'why', 'can', 'not', 'have', 'nice', ...",7,6,this why can not have nice things


['date', 'user_id', 'user_name', 'text', 'raw_post_name', 'post_id_true', 'post_text', 'post_description', 'post_link', 'post_page_id', 'post_created_time', 'post_scrape_time', 'post_shares', 'post_react_angry', 'post_react_haha', 'post_react_like', 'post_react_love', 'post_react_sad', 'post_react_wow', 'raw_post_id', 'has_post_context', 'split', 'normalized_text', 'clean_tokens', 'clean_token_count', 'informative_token_count', 'clean_text']


<div class="nb-section">
  <h2>07 — Select Text Column</h2>

  <p>
    تحدد هذه الخلية عمود النص المستخدم في التقييم.
    العمود المعتمد هو:
    <span class="ltr">clean_text</span>.
  </p>

  <p>
    يتم حذف السجلات الفارغة فقط، ثم تُحوَّل النصوص إلى قائمة
    <span class="ltr">docs</span>
    التي سيتم تمريرها إلى نموذج
    <span class="ltr">SentenceTransformer</span>
    لتوليد
    <span class="ltr">embeddings</span>.
  </p>

  <div class="warning">
    بعد توليد
    <span class="ltr">embeddings</span>
    يجب عدم تغيير ترتيب الصفوف أو حذف سجلات جديدة، لأن كل
    <span class="ltr">embedding</span>
    يجب أن يبقى مرتبطًا بالتعليق الصحيح.
  </div>
</div>

In [ ]:
TEXT_COL = "clean_text"

if TEXT_COL not in df.columns:
    raise ValueError(f"Column '{TEXT_COL}' not found. Available columns: {df.columns.tolist()}")

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()

# Remove empty texts
df = df[df[TEXT_COL].ne("")].copy().reset_index(drop=True)

docs = df[TEXT_COL].tolist()

print("Shape after removing empty texts:", df.shape)
print("Number of documents:", len(docs))

display(df[[TEXT_COL]].head())

Shape after removing empty texts: (58014, 27)
Number of documents: 58014


,clean_text
0,they ransacked the prisons and freed them that...
1,stupidity what alex relies sell his products f...
2,silver once one the most powerful people new y...
3,democrats have placed such emphasis and relied...
4,this why can not have nice things


<div class="nb-section">
  <h2>08 — Light English Text Cleaning</h2>

  <p>
    تطبّق هذه الخلية تنظيفًا خفيفًا على النصوص الإنجليزية قبل تمريرها إلى النموذج.
  </p>

  <p>
    يشمل التنظيف تحويل النص إلى أحرف صغيرة، استبدال الروابط، البريد الإلكتروني،
    والأرقام برموز عامة، ثم إزالة الرموز غير الضرورية وتوحيد المسافات.
  </p>

  <div class="note">
    هذا التنظيف لا يغيّر المعنى العام للتعليقات، بل يقلّل الضجيج النصي قبل توليد
    <span class="ltr">embeddings</span>.
  </div>
</div>

In [ ]:
def light_clean_english_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " url ", text)
    text = re.sub(r"\S+@\S+", " email ", text)
    text = re.sub(r"\d+", " number ", text)
    text = re.sub(r"[^a-zA-Z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["model_text"] = df[TEXT_COL].apply(light_clean_english_text)

df = df[df["model_text"].str.len() > 2].copy()

print("Shape after light cleaning:", df.shape)
display(df[["model_text"]].head())

Shape after light cleaning: (58014, 30)


,model_text
0,they ransacked the prisons and freed them that...
1,stupidity what alex relies sell his products f...
2,silver once one the most powerful people new y...
3,democrats have placed such emphasis and relied...
4,this why can not have nice things


<div class="nb-section">
  <h2>09 — Dataset Statistics</h2>

  <p>
    تعرض هذه الخلية إحصاءات وصفية عن النصوص المستخدمة في التقييم، مثل عدد الوثائق،
    متوسط عدد الحروف، متوسط عدد الكلمات، وأطول تعليق.
  </p>

  <p>
    تساعد هذه الإحصاءات على توثيق طبيعة بيانات التعليقات قبل تمريرها إلى النموذج.
  </p>
</div>

In [ ]:
df["text_len"] = df["model_text"].apply(len)
df["word_count"] = df["model_text"].apply(lambda x: len(x.split()))

print("Documents:", len(df))
print("Average characters:", round(df["text_len"].mean(), 2))
print("Average words:", round(df["word_count"].mean(), 2))
print("Min words:", df["word_count"].min())
print("Max words:", df["word_count"].max())

display(df[["text_len", "word_count"]].describe())

Documents: 58014
Average characters: 159.32
Average words: 26.45
Min words: 3
Max words: 1096


,text_len,word_count
count,58014.000000,58014.000000
mean,159.320354,26.449598
std,262.311100,41.427199
min,10.000000,3.000000
25%,52.000000,9.000000
50%,98.000000,17.000000
75%,182.000000,30.000000
max,6955.000000,1096.000000


<div class="nb-section">
  <h2>10 — Load Final Embedding Model</h2>

  <p>
    تقوم هذه الخلية بتحميل النموذج النهائي المعتمد:
    <span class="ltr">Final Fine-Tuned MiniLM Comment Encoder</span>.
  </p>

  <p>
    يستخدم النموذج لإنتاج
    <span class="ltr">embedding vector</span>
    لكل تعليق، بحيث يمثل هذا المتجه المعنى الدلالي للتعليق قبل تمريره إلى
    <span class="ltr">UMAP</span>
    و
    <span class="ltr">HDBSCAN</span>
    داخل خط
    <span class="ltr">BERTopic</span>.
  </p>

  <p>
    يتم ضبط:
    <span class="metric">max_seq_length = 256</span>
    لضمان اتساق التقييم مع إعدادات النموذج النهائي.
  </p>
</div>

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(str(MODEL_PATH), device=device)
embedding_model.max_seq_length = 256

print("Loaded model:", MODEL_PATH)
print("Device:", device)
print("Max sequence length:", embedding_model.max_seq_length)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded model: /content/drive/MyDrive/semester project/news_comment_topic_system/models/v7_fullft_minilm_raw_largepairs_1p5M_seq256/final_model
Device: cuda
Max sequence length: 256
GPU: NVIDIA A100-SXM4-40GB


<div class="nb-section">
  <h2>11 — Generate Comment Embeddings</h2>

  <p>
    في هذه الخطوة يتم استخدام النموذج النهائي
    <span class="ltr">Final Fine-Tuned MiniLM Comment Encoder</span>
    لتحويل كل تعليق إلى
    <span class="ltr">embedding vector</span>.
  </p>

  <p>
    يتم توليد
    <span class="ltr">embeddings</span>
    من عمود
    <span class="ltr">model_text</span>
    بعد مرحلة التنظيف الخفيف للنصوص.
    هذه المتجهات ستمثل المعنى الدلالي للتعليقات، وسيتم استخدامها لاحقًا في
    <span class="ltr">UMAP</span>
    و
    <span class="ltr">HDBSCAN</span>
    داخل خط
    <span class="ltr">BERTopic</span>.
  </p>

  <div class="note">
    الناتج المتوقع هو مصفوفة
    <span class="ltr">embeddings</span>
    بعدد صفوف يساوي عدد التعليقات، وبعدد أبعاد يساوي
    <span class="metric">384</span>.
  </div>
</div>

In [ ]:
docs = df["model_text"].tolist()

embeddings = embedding_model.encode(
    docs,
    batch_size=512,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)
print("Documents:", len(docs))

assert embeddings.shape[0] == len(docs), "Mismatch between embeddings and documents."

Batches:   0%|          | 0/114 [00:00<?, ?it/s]

Saved embeddings: /content/drive/MyDrive/semester project/news_comment_topic_system/bertopic_experiments_clean/v7_fullft_minilm_raw_largepairs_1p5M_seq256/embeddings_v7_seq256_test_large.npy
Embeddings shape: (58014, 384)


<div class="nb-section">
  <h2>12 — Save Embeddings Cache</h2>

  <p>
    تحفظ هذه الخلية مصفوفة
    <span class="ltr">embeddings</span>
    الناتجة في ملف
    <span class="ltr">.npy</span>
    داخل مجلد المخرجات النهائي.
  </p>

  <p>
    الهدف من حفظ الملف هو توثيق ناتج هذه المرحلة وإمكانية إعادة استخدامه لاحقًا
    دون الحاجة إلى توليد
    <span class="ltr">embeddings</span>
    من جديد.
  </p>

  <div class="warning">
    هذه الخلية لا تولّد
    <span class="ltr">embeddings</span>
    ولا تعيد تحميلها، بل تحفظ الناتج الموجود من الخلية السابقة فقط.
  </div>
</div>

In [ ]:
import numpy as np

EMBEDDINGS_PATH = OUTPUT_DIR / "test_large_embeddings.npy"

np.save(EMBEDDINGS_PATH, embeddings)

print("Saved embeddings to:", EMBEDDINGS_PATH)
print("Embeddings shape:", embeddings.shape)

Saved embeddings to: /content/drive/MyDrive/semester project/news_comment_topic_system/bertopic_v6b_seq256_evaluation_outputs/test_large_embeddings.npy


<div dir="rtl" style="text-align:right; line-height:1.8; font-size:16px;">

<h2>13 - UMAP Configuration</h2>

<p>
في هذه الخلية نقوم بإعداد نموذج
<span dir="ltr">UMAP</span>
المستخدم لتقليل أبعاد
<span dir="ltr">embeddings</span>
قبل مرحلة التجميع.
</p>

<p>
تقليل الأبعاد يساعد
<span dir="ltr">HDBSCAN</span>
على اكتشاف العناقيد الدلالية بشكل أفضل.
</p>

</div>

In [ ]:
from umap import UMAP

UMAP_N_NEIGHBORS = 15
UMAP_N_COMPONENTS = 5
UMAP_MIN_DIST = 0.0
UMAP_METRIC = "cosine"
RANDOM_STATE = 42

umap_model = UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    n_components=UMAP_N_COMPONENTS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=RANDOM_STATE
)

print("UMAP configured.")

UMAP configured.


<div dir="rtl" style="text-align:right; line-height:1.8; font-size:16px;">

<h2>14 - HDBSCAN Configuration</h2>

<p>
في هذه الخلية نقوم بإعداد نموذج
<span dir="ltr">HDBSCAN</span>
المستخدم لاكتشاف العناقيد داخل فضاء
<span dir="ltr">embeddings</span>
بعد تقليل الأبعاد.
</p>

<p>
كل عنقود يتم اكتشافه لاحقًا يمثل موضوعًا محتملاً، أما التعليقات التي لا تنتمي إلى عنقود واضح فتأخذ القيمة
<span dir="ltr">-1</span>
وتُعامل كضجيج أو
<span dir="ltr">Noise</span>.
</p>

</div>

In [ ]:
from hdbscan import HDBSCAN

HDBSCAN_MIN_CLUSTER_SIZE = 100
HDBSCAN_MIN_SAMPLES = 10
HDBSCAN_METRIC = "euclidean"

hdbscan_model = HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    metric=HDBSCAN_METRIC,
    cluster_selection_method="eom",
    prediction_data=True
)

print("HDBSCAN configured.")

HDBSCAN configured.


<div class="nb-section">
  <h2>15 — English Stopwords and Vectorizer</h2>

  <p>
    في هذه الخلية نستخدم مكتبة
    <span class="ltr">NLTK</span>
    لتحميل قائمة
    <span class="ltr">English stopwords</span>
    لأن نصوص التقييم باللغة الإنجليزية.
  </p>

  <p>
    بعد ذلك نستخدم
    <span class="ltr">CountVectorizer</span>
    لاستخراج الكلمات والعبارات المرشحة التي سيستخدمها
    <span class="ltr">BERTopic</span>
    لاحقًا في تمثيل المواضيع عبر
    <span class="ltr">c-TF-IDF</span>.
  </p>

  <p>
    هذه المرحلة لا تغيّر نتيجة التجميع مباشرة، لكنها تؤثر على جودة الكلمات التي تظهر
    لشرح كل موضوع.
  </p>
</div>

In [ ]:

nltk.download("stopwords", quiet=True)

english_stopwords = stopwords.words("english")

print("Number of English stopwords:", len(english_stopwords))
print("Sample stopwords:", english_stopwords[:20])

vectorizer_model = CountVectorizer(
    stop_words=english_stopwords,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.7,
    analyzer="word"
)

Number of English stopwords: 198
Sample stopwords: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']


<div class="nb-section">
  <h2>16 — Build BERTopic Model</h2>

  <p>
    في هذه الخلية يتم بناء نموذج
    <span class="ltr">BERTopic</span>
    من خلال ربط المكونات التي تم إعدادها سابقًا:
    <span class="ltr">UMAP</span>،
    <span class="ltr">HDBSCAN</span>،
    و
    <span class="ltr">CountVectorizer</span>.
  </p>

  <p>
    يتم استخدام
    <span class="ltr">embedding_model=None</span>
    لأن
    <span class="ltr">embeddings</span>
    تم توليدها مسبقًا باستخدام النموذج النهائي، وسيتم تمريرها مباشرة إلى
    <span class="ltr">fit_transform</span>.
  </p>
</div>

In [ ]:
topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    verbose=True
)

print("BERTopic model configured.")

<div class="nb-section">
  <h2>17 — Fit BERTopic and Assign Topics</h2>

  <p>
    في هذه الخلية يتم تشغيل نموذج
    <span class="ltr">BERTopic</span>
    على النصوص وتمثيلات
    <span class="ltr">embeddings</span>
    الناتجة من النموذج النهائي.
  </p>

  <p>
    الناتج الأساسي هو رقم الموضوع
    <span class="ltr">topic</span>
    لكل تعليق، إضافة إلى قيم الثقة
    <span class="ltr">probs</span>
    التي سيتم استخدامها لاحقًا في حساب المقاييس.
  </p>

  <p>
    يتم إسناد رقم الموضوع إلى
    <span class="ltr">DataFrame</span>
    حتى يمكن لاحقًا تحليل التعليقات حسب الموضوع.
  </p>
</div>

In [3]:
topics, probs = topic_model.fit_transform(docs, embeddings)

df["topic"] = topics

print("Done.")
print("Number of documents:", len(df))
print("Unique topics including -1:", len(set(topics)))
print("Unique valid topics:", len(set(t for t in topics if t != -1)))

NameError: name 'topic_model' is not defined

<div class="nb-section">
  <h2>18 — Topic Information Table</h2>

  <p>
    تعرض هذه الخلية جدول معلومات المواضيع باستخدام
    <span class="ltr">get_topic_info</span>.
  </p>

  <p>
    يحتوي الجدول على رقم كل موضوع، عدد التعليقات داخله، اسمه التلقائي،
    والكلمات التي تمثله.
  </p>

  <p>
    يساعد هذا الجدول على فهم البنية العامة للمواضيع قبل الانتقال إلى حساب المقاييس
    والفحص اليدوي.
  </p>
</div>

In [ ]:
topic_info = topic_model.get_topic_info()

display(topic_info.head(20))

TOPIC_INFO_PATH = OUTPUT_DIR / "final_topic_info.csv"

topic_info.to_csv(TOPIC_INFO_PATH, index=False)

print("Saved topic info:", TOPIC_INFO_PATH)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,18556,-1_republicans_obama_gop_kids,"[republicans, obama, gop, kids, child, law, de...","[administration very dirty collusion obvious, ..."
1,0,3603,0_russia_putin_russian_russians,"[russia, putin, russian, russians, meeting, el...","[and that why does not care what the russians,..."
2,1,2331,1_woman_lady_first lady_wife,"[woman, lady, first lady, wife, women, ivanka,...",[our first lady beautiful and classy unlike th...
3,2,2315,2_president trump_donald trump_white house_trumps,"[president trump, donald trump, white house, t...","[well trump supporters there you, love preside..."
4,3,1748,3_tax_taxes_rich_jobs,"[tax, taxes, rich, jobs, wealthy, market, wage...",[likely got big refund but nobody has seen the...
5,4,1620,4_healthcare_insurance_health_health care,"[healthcare, insurance, health, health care, o...",[the aca not health insurance plan law governi...
6,5,926,5_religion_christian_church_bible,"[religion, christian, church, bible, jesus, re...",[this does not account for the fact that even ...
7,6,907,6_fake news_press_reporters_journalists,"[fake news, press, reporters, journalists, jou...","[some more fake news, fake news again, more fa..."
8,7,805,7_food_chocolate_eat_sugar,"[food, chocolate, eat, sugar, burger, fat, not...","[nothing burger news, this such nothing burger..."
9,8,792,8_late_surprise_awesome_surprised,"[late, surprise, awesome, surprised, sounds ri...","[few years too late maybe, years too late, too..."


Saved topic info: /content/drive/MyDrive/semester project/news_comment_topic_system/bertopic_experiments_clean/v7_fullft_minilm_raw_largepairs_1p5M_seq256/v7_seq256_umap15_mcs100_ms10/topic_info_v7_seq256_umap15_mcs100_ms10.csv


<div class="nb-section">
  <h2>19 — Core Evaluation Metrics</h2>

  <p>
    تحسب هذه الخلية المقاييس الأساسية لتقييم سلوك
    <span class="ltr">BERTopic</span>
    على بيانات التعليقات.
  </p>

  <p>
    لأن المشروع يعتمد على
    <span class="ltr">Topic Modeling</span>
    وليس على
    <span class="ltr">Supervised Classification</span>،
    لا نستخدم مقاييس مثل
    <span class="ltr">accuracy</span>
    أو
    <span class="ltr">precision</span>
    التقليدية.
  </p>

  <p>
    بدلًا من ذلك، نقيّم عدد التعليقات التي أُسندت إلى مواضيع صالحة،
    نسبة التعليقات المصنفة كضجيج،
    عدد المواضيع المكتشفة،
    متوسط الثقة،
    وتنوع كلمات المواضيع.
  </p>
</div>

In [ ]:
import numpy as np
import pandas as pd

df["topic"] = topics

# Robust probability handling
if probs is None:
    df["prob"] = np.nan
else:
    probs_array = np.asarray(probs)

    print("probs shape:", probs_array.shape)
    print("probs ndim:", probs_array.ndim)

    if probs_array.ndim == 1:
        # One confidence value per document
        df["prob"] = probs_array
    elif probs_array.ndim == 2:
        # Full topic-probability matrix
        df["prob"] = np.max(probs_array, axis=1)
    else:
        raise ValueError(f"Unexpected probs shape: {probs_array.shape}")

total_docs = len(df)
noise_docs = int((df["topic"] == -1).sum())
valid_docs = total_docs - noise_docs

valid_ratio = round(valid_docs / total_docs * 100, 2)
noise_ratio = round(noise_docs / total_docs * 100, 2)

detected_topics = len(set(t for t in topics if t != -1))

avg_conf_all = round(float(df["prob"].mean() * 100), 2)
avg_conf_valid = round(float(df.loc[df["topic"] != -1, "prob"].mean() * 100), 2)
median_conf_valid = round(float(df.loc[df["topic"] != -1, "prob"].median() * 100), 2)

topic_info = topic_model.get_topic_info()

def compute_topic_diversity(topic_model, top_n=10):
    words = []

    for topic_id in topic_model.get_topics().keys():
        if topic_id == -1:
            continue

        topic_words = topic_model.get_topic(topic_id)
        if topic_words is None:
            continue

        words.extend([word for word, score in topic_words[:top_n]])

    if len(words) == 0:
        return 0.0

    return round(len(set(words)) / len(words), 4)

topic_diversity = compute_topic_diversity(topic_model, top_n=10)

metrics_df = pd.DataFrame([{
    "Model": FINAL_MODEL_DISPLAY_NAME,
    "Experiment": FINAL_EXPERIMENT_NAME,

    "Total Documents": total_docs,
    "Valid Topic Comments": valid_docs,
    "Noise Comments": noise_docs,
    "Valid Ratio": valid_ratio,
    "Noise Ratio": noise_ratio,
    "Detected Topics": detected_topics,

    "Average Confidence All": avg_conf_all,
    "Average Confidence Valid": avg_conf_valid,
    "Median Confidence Valid": median_conf_valid,
    "Topic Diversity": topic_diversity,

    "UMAP n_neighbors": UMAP_N_NEIGHBORS,
    "UMAP n_components": UMAP_N_COMPONENTS,
    "UMAP min_dist": UMAP_MIN_DIST,
    "UMAP metric": UMAP_METRIC,

    "HDBSCAN min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
    "HDBSCAN min_samples": HDBSCAN_MIN_SAMPLES,
    "HDBSCAN metric": HDBSCAN_METRIC,

    "Vectorizer ngram_range": "(1, 2)",
    "Vectorizer min_df": 2,
    "Vectorizer max_df": 0.7
}])

display(metrics_df.T)
display(topic_info.head(20))

probs shape: (58014,)
probs ndim: 1


,Model,Experiment,Total Documents,Valid Topic Comments,Noise Comments,Valid Ratio,Noise Ratio,Detected Topics,Average Confidence All,Average Confidence Valid,...,UMAP n_neighbors,UMAP n_components,UMAP min_dist,UMAP metric,HDBSCAN min_cluster_size,HDBSCAN min_samples,HDBSCAN metric,Vectorizer ngram_range,Vectorizer min_df,Vectorizer max_df
0,V7 FullFT MiniLM RawLargePairs Seq256,v7_seq256_umap15_mcs100_ms10,58014,39458,18556,68.01,31.99,94,56.13,82.52,...,15,5,0.0,cosine,100,10,euclidean,"(1, 2)",2,0.7


,Topic,Count,Name,Representation,Representative_Docs
0,-1,18556,-1_republicans_obama_gop_kids,"[republicans, obama, gop, kids, child, law, de...","[administration very dirty collusion obvious, ..."
1,0,3603,0_russia_putin_russian_russians,"[russia, putin, russian, russians, meeting, el...","[and that why does not care what the russians,..."
2,1,2331,1_woman_lady_first lady_wife,"[woman, lady, first lady, wife, women, ivanka,...",[our first lady beautiful and classy unlike th...
3,2,2315,2_president trump_donald trump_white house_trumps,"[president trump, donald trump, white house, t...","[well trump supporters there you, love preside..."
4,3,1748,3_tax_taxes_rich_jobs,"[tax, taxes, rich, jobs, wealthy, market, wage...",[likely got big refund but nobody has seen the...
5,4,1620,4_healthcare_insurance_health_health care,"[healthcare, insurance, health, health care, o...",[the aca not health insurance plan law governi...
6,5,926,5_religion_christian_church_bible,"[religion, christian, church, bible, jesus, re...",[this does not account for the fact that even ...
7,6,907,6_fake news_press_reporters_journalists,"[fake news, press, reporters, journalists, jou...","[some more fake news, fake news again, more fa..."
8,7,805,7_food_chocolate_eat_sugar,"[food, chocolate, eat, sugar, burger, fat, not...","[nothing burger news, this such nothing burger..."
9,8,792,8_late_surprise_awesome_surprised,"[late, surprise, awesome, surprised, sounds ri...","[few years too late maybe, years too late, too..."


In [ ]:
METRICS_PATH = OUTPUT_DIR / "final_metrics.csv"
TOPIC_INFO_PATH = OUTPUT_DIR / "final_topic_info.csv"
ASSIGNMENTS_PATH = OUTPUT_DIR / "final_topic_assignments.csv"

metrics_df.to_csv(METRICS_PATH, index=False)
topic_info.to_csv(TOPIC_INFO_PATH, index=False)
df.to_csv(ASSIGNMENTS_PATH, index=False)

print("Saved metrics:", METRICS_PATH)
print("Saved topic info:", TOPIC_INFO_PATH)
print("Saved assignments:", ASSIGNMENTS_PATH)

Saved metrics: /content/drive/MyDrive/semester project/news_comment_topic_system/bertopic_experiments_clean/v7_fullft_minilm_raw_largepairs_1p5M_seq256/v7_seq256_umap15_mcs100_ms10/metrics_v7_seq256_umap15_mcs100_ms10.csv
Saved topic info: /content/drive/MyDrive/semester project/news_comment_topic_system/bertopic_experiments_clean/v7_fullft_minilm_raw_largepairs_1p5M_seq256/v7_seq256_umap15_mcs100_ms10/topic_info_v7_seq256_umap15_mcs100_ms10.csv
Saved assignments: /content/drive/MyDrive/semester project/news_comment_topic_system/bertopic_experiments_clean/v7_fullft_minilm_raw_largepairs_1p5M_seq256/v7_seq256_umap15_mcs100_ms10/topic_assignments_v7_seq256_umap15_mcs100_ms10.csv


<div class="nb-section">
  <h2>20 — Topic Diversity Interpretation</h2>

  <p>
    تعرض هذه الخلية قيمة
    <span class="ltr">Topic Diversity</span>
    المحسوبة ضمن المقاييس الأساسية.
  </p>

  <p>
    تقيس هذه القيمة مدى تنوع الكلمات الممثلة للمواضيع. كلما اقتربت القيمة من
    <span class="metric">1.0</span>
    كان تكرار الكلمات بين المواضيع أقل، وكانت المواضيع أكثر تميّزًا من حيث التمثيل اللفظي.
  </p>
</div>

In [ ]:
display(metrics_df[["Model", "Experiment", "Topic Diversity"]])

Topic Diversity: 0.9011


<div dir="rtl" style="text-align:right; line-height:1.8; font-size:16px;">

<h2>21 - Final Metrics Summary</h2>

<p>
في هذه الخلية نجمع المقاييس النهائية والبارامترات الأساسية في جدول واحد.
</p>

<p>
هذا الجدول مناسب للعرض في التقرير أو أمام اللجنة لأنه يلخص أداء النموذج وإعدادات
<span dir="ltr">BERTopic</span>
المستخدمة.
</p>

</div>

In [ ]:
# Compact Final Metrics Summary

compact_summary = metrics_df[[
    "Model",
    "Experiment",
    "Total Documents",
    "Valid Topic Comments",
    "Noise Comments",
    "Noise Ratio",
    "Detected Topics",
    "Average Confidence Valid",
    "Median Confidence Valid",
    "Topic Diversity"
]].copy()

display(compact_summary.T)

<div class="nb-section">
  <h2>22 — Manual Topic Inspection</h2>

  <p>
    في هذه المرحلة نفحص بعض المواضيع يدويًا للتأكد من أن النتائج مفهومة دلاليًا.
    نعرض كلمات الموضوع، ثم بعض التعليقات التابعة له.
  </p>

  <p>
    هذا الفحص يساعد على دعم المقاييس الرقمية بقراءة فعلية لمحتوى المواضيع.
  </p>
</div>

In [ ]:
TOPIC_ID = 0

print("Topic ID:", TOPIC_ID)
print("Number of comments:", len(df[df["topic"] == TOPIC_ID]))

print("\nTop words:")
display(pd.DataFrame(topic_model.get_topic(TOPIC_ID), columns=["word", "score"]))

print("\nSample comments:")
display(
    df[df["topic"] == TOPIC_ID][["model_text", "topic", "prob"]]
    .sample(10, random_state=42)
)

<div dir="rtl" style="text-align:right; line-height:1.8; font-size:16px;">

<h2>23 - Inspect Noise Comments</h2>

<p>
في هذه الخلية نفحص التعليقات التي حصلت على القيمة
<span dir="ltr">-1</span>
.
</p>

<p>
هذه التعليقات لم يتمكن
<span dir="ltr">HDBSCAN</span>
من ربطها بموضوع واضح، لذلك تُعامل كضجيج أو
<span dir="ltr">Noise</span>.
</p>

</div>

In [ ]:
noise_comments = df[df["topic"] == -1]

print("Number of noise comments:", len(noise_comments))

display(
    noise_comments[["model_text", "topic", "prob"]]
    .sample(10, random_state=42)
)

<div dir="rtl" style="text-align:right; line-height:1.8; font-size:16px;">

<h2>24 - Intertopic Distance Map</h2>

<p>
في هذه الخلية نعرض خريطة المسافة بين المواضيع.
</p>

<p>
كل دائرة تمثل موضوعًا، وحجم الدائرة يعكس حجم الموضوع تقريبًا، بينما قرب الدوائر من بعضها يشير إلى تشابه المواضيع.
</p>

<p>
هذه الخريطة استكشافية ولا تعتبر مقياسًا كميًا نهائيًا.
</p>

</div>

In [ ]:
fig = topic_model.visualize_topics()
fig

<div dir="rtl" style="text-align:right; line-height:1.8; font-size:16px;">

<h2>25 - Topic Bar Chart</h2>

<p>
في هذه الخلية نعرض رسمًا شريطيًا لأهم المواضيع والكلمات التي تمثلها.
</p>

<p>
هذا الرسم يساعد على فحص المواضيع الأكثر بروزًا داخل البيانات.
</p>

</div>

In [ ]:
fig = topic_model.visualize_barchart(top_n_topics=20)
fig

<div dir="rtl" style="text-align:right; line-height:1.8; font-size:16px;">

<h2>26 - Document Map Visualization</h2>

<p>
في هذه الخلية نعرض خريطة بصرية للتعليقات حسب المواضيع.
</p>

<p>
هذه الخريطة تساعد على رؤية كيفية توزع التعليقات والعناقيد، لكنها قد تكون ثقيلة عند استخدام عدد كبير من الوثائق.
</p>

</div>

In [ ]:
fig = topic_model.visualize_documents(
    docs,
    embeddings=embeddings,
    sample=0.1,
    hide_document_hover=True
)

fig

<div dir="rtl" style="text-align:right; line-height:1.8; font-size:16px;">

<h2>27 - Detect Date Column</h2>

<p>
في هذه الخلية نحاول تحديد عمود التاريخ الموجود في البيانات.
</p>

<p>
وجود التاريخ ضروري إذا أردنا تحليل تطور المواضيع عبر الزمن باستخدام
<span dir="ltr">topics_over_time</span>.
</p>

</div>

In [ ]:
# Detect date column for Topics Over Time

possible_date_cols = [
    "created_time",
    "scrape_time",
    "date",
    "created_at",
    "datetime",
    "timestamp",
    "post_date",
    "comment_date"
]

DATE_COL = None

for col in possible_date_cols:
    if col in df.columns:
        DATE_COL = col
        break

print("Detected DATE_COL:", DATE_COL)

if DATE_COL is None:
    print("No date column found. Topics over time will be skipped.")
else:
    display(df[[DATE_COL]].head())

Detected DATE_COL: date


<div dir="rtl" style="text-align:right; line-height:1.8; font-size:16px;">

<h2>29 - Daily Topics Over Time</h2>

<p>
في هذه الخلية نحسب تطور المواضيع عبر الزمن على مستوى اليوم فقط.
</p>

<p>
نحوّل كل تاريخ إلى بداية اليوم حتى لا يتم الحساب على مستوى الساعة أو الدقيقة أو الثانية.
</p>

<p>
هذا يجعل التحليل الزمني أكثر وضوحًا، ويتيح معرفة الأيام التي ارتفع فيها موضوع معين بشكل حاد.
</p>

</div>

In [ ]:
if DATE_COL is None:
    print("No date column found. Topics Over Time skipped.")

else:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

    time_df = df.dropna(subset=[DATE_COL]).copy()
    time_df["date_day"] = time_df[DATE_COL].dt.floor("D")

    docs_time = time_df["model_text"].tolist()
    timestamps = time_df["date_day"].tolist()
    topics_time = time_df["topic"].tolist()

    topics_over_time = topic_model.topics_over_time(
        docs=docs_time,
        timestamps=timestamps,
        topics=topics_time
    )

    display(topics_over_time.head())

    TOPICS_OVER_TIME_PATH = OUTPUT_DIR / "final_topics_over_time_by_day.csv"
    topics_over_time.to_csv(TOPICS_OVER_TIME_PATH, index=False)

    print("Saved:", TOPICS_OVER_TIME_PATH)

Documents with valid dates: 58014
Daily timestamps: 58014
Unique days: 372
Topics: 58014


2026-04-28 22:30:45,406 - BERTopic - WARNING: There are more than 100 unique timestamps (i.e., 372) which significantly slows down the application. Consider setting `nr_bins` to a value lower than 100 to speed up calculation. 
372it [01:16,  4.85it/s]


,Topic,Words,Frequency,Timestamp
0,-1,"huckabee, gov huckabee, gov, premarital sex, p...",18,2015-01-29 00:00:00+00:00
1,2,"pick apart, stay strong, would pick, knew woul...",1,2015-01-29 00:00:00+00:00
2,11,"use say, place great, great grandma, teachers ...",1,2015-01-29 00:00:00+00:00
3,13,"language well, intelligent way, would correct,...",1,2015-01-29 00:00:00+00:00
4,29,"men job, like women, want women, women, women ...",2,2015-01-29 00:00:00+00:00


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/bertopic_v6b_seq256_evaluation_outputs/topics_over_time_daily_v5_stage1.csv


<div dir="rtl" style="text-align:right; line-height:1.8; font-size:16px;">

<h2>30 - Visualize Daily Topics Over Time</h2>

<p>
في هذه الخلية نعرض الرسم الزمني للمواضيع الأكثر ظهورًا.
</p>

<p>
يساعد هذا الرسم على كشف الارتفاعات المفاجئة
<span dir="ltr">spikes</span>
في بعض المواضيع، ثم يمكن الرجوع إلى التعليقات في ذلك اليوم لمعرفة الحدث أو النقاش الذي سبب الارتفاع.
</p>

</div>

In [ ]:
import plotly.io as pio

pio.renderers.default = "colab"

valid_topics = topics_over_time[topics_over_time["Topic"] != -1]

top_topics = (
    valid_topics.groupby("Topic")["Frequency"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
    .tolist()
)

print("Selected top topics:", top_topics)

fig = topic_model.visualize_topics_over_time(
    topics_over_time,
    topics=top_topics,
    normalize_frequency=False,
    width=1200,
    height=650
)

fig.show()

Selected topics: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Total frequency of selected topics:


,Frequency
Topic,
0,3287
1,1853
2,1310
3,1277
4,903
5,650
6,647
7,513
8,498


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

FINAL_EVAL_DIR = (
    PROJECT_ROOT
    / "final_bertopic_evaluation_outputs"
    / "final_minilm_comment_encoder_umap15_mcs100_ms10"
)

FINAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)

BERTOPIC_MODEL_PATH = FINAL_EVAL_DIR / "final_bertopic_model"

topic_model.save(
    str(BERTOPIC_MODEL_PATH),
    serialization="pickle"
)

print("Saved BERTopic model to:", BERTOPIC_MODEL_PATH)

NameError: name 'topic_model' is not defined